# Notebook 34 — Reconstruct the RMM index from Barlow-Twins latents (phase / envelope decomposition)
**Project:** ENSO-BSISO SSL — MJO moisture-constraint extension
**Author:** Jiayi (jh9141@nyu.edu)

**Purpose.** Quantify *which* component of the operational MJO index (RMM) the invariance-trained
**Barlow-Twins** latent captures, by reconstructing RMM and scoring its **amplitude** and **phase**
parts **separately**. Prior (nb33 / summary sec 8): Barlow is phase-blind but holds a slow envelope
-> expected **amplitude recoverable, phase ~ chance**. This restates sec 8 against the canonical,
interpretable index (not raw fields).

**Design (locked with user, Session 59):**
- Target = own-RMM continuous PCs (primary) + official BoM RMM (robustness check).
- Scoring = decompose into amplitude (R2) and phase (circular corr / 8-sector acc / angular error).
- Probes = linear (Ridge) **and** nonlinear (MLP) — MLP rules out "phase present but nonlinearly hidden".
- Latents = Barlow-D3 and Barlow-D7 only; own-RMM-self as a pipeline ceiling; analytic chance refs.

**Key alignment note.** Barlow embeddings live on the **bp20-90 axis**; own-RMM + BoM live on the
**nb30 `labels_aligned_mjo` axis**. Cell 2 date-aligns them by intersection (assert overlap).

## Cell 1 — Setup, load, and DATE-ALIGN Barlow (bp axis) to RMM (ref axis)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, balanced_accuracy_score

PROJECT_DIR='/content/drive/MyDrive/BSISO_SSL_Project'; MJO_DIR=f'{PROJECT_DIR}/MJO'
PROC=f'{MJO_DIR}/data/processed'
OUT=f'{MJO_DIR}/moisture_constraints/results/barlow_rmm'; os.makedirs(OUT, exist_ok=True)

# --- reference axis: own-RMM + official BoM both live here (nb30 axis) ---
lab_ref=pd.read_csv(f'{PROC}/labels_aligned_mjo.csv', parse_dates=['date'])
own=np.load(f'{PROC}/mjo_rmm_own_pcs.npy').astype(np.float32)     # (Nref,2) rotation-aligned to BoM
assert len(lab_ref)==own.shape[0], 'own-RMM vs ref-labels length mismatch'
dref=pd.DatetimeIndex(lab_ref['date']).normalize()

# --- Barlow embeddings live on the bp20-90 axis ---
lab_bp=pd.read_csv(f'{PROC}/labels_aligned_mjo_bp20_90.csv', parse_dates=['date'])
dbp=pd.DatetimeIndex(lab_bp['date']).normalize()
BARLOW={'Barlow-D3': f'{MJO_DIR}/barlow/D3/embeddings_z7.npy',
        'Barlow-D7': f'{MJO_DIR}/barlow/embeddings_z7.npy'}

# --- date alignment: map each bp row to its ref row ---
ref_pos={d:i for i,d in enumerate(dref)}
keep_bp=np.array([d in ref_pos for d in dbp])
ref_idx=np.array([ref_pos[d] for d in dbp[keep_bp]])
print(f'bp days {len(dbp)}  ref days {len(dref)}  aligned (bp found in ref) {int(keep_bp.sum())}')
assert keep_bp.sum() > 1000, 'too few aligned days -- check axes'

# --- targets on the aligned subset ---
own_k=own[ref_idx]; RMM1,RMM2=own_k[:,0],own_k[:,1]
A_true=np.sqrt(RMM1**2+RMM2**2); theta_true=np.arctan2(RMM2,RMM1)
bom_k=lab_ref[['rmm1','rmm2']].values[ref_idx].astype(np.float32)     # official BoM, same rows
phase_lbl=lab_ref['phase'].values[ref_idx].astype(int)
amp_lbl=lab_ref['amplitude'].values[ref_idx].astype(float)
weak=lab_ref['weak_mjo'].values[ref_idx].astype(bool)
dates_k=dbp[keep_bp]; years=dates_k.year.values

# --- active-day mask + year-based split (every 5th year = val) ---
active=(~weak)&(amp_lbl>=1.0)
is_val=np.isin(years, sorted(np.unique(years))[::5]); tr=(~is_val); va=is_val
print(f'active MJO days {int(active.sum())}   train&active {int((tr&active).sum())}   val&active {int((va&active).sum())}')

def load_barlow(name):
    p=BARLOW[name]
    if not os.path.exists(p): print('[skip missing]',name); return None
    Z=np.load(p).astype(np.float32)
    if Z.shape[0]!=len(dbp): print(f'[skip shape] {name} {Z.shape} vs bp {len(dbp)}'); return None
    return Z[keep_bp]     # -> aligned to ref subset (row-matched to targets)

## Cell 2 — Probe helpers (linear + MLP), phase & amplitude scored separately

- **Phase probe:** fit latent -> `(cos theta, sin theta)`, recover `theta_hat = atan2(sin_hat, cos_hat)`.
  Metrics: **circular correlation** (offset-invariant, primary), **8-sector balanced accuracy**
  (self-consistent octants from atan2), **mean absolute angular error (deg)**.
- **Amplitude probe:** fit latent -> `A`, score **R2**.
- Both fit on train&active, evaluate on val&active, latent standardized on train.

In [ ]:
def circ_corr(a,b):                        # Jammalamadaka-Sarma circular correlation (radians)
    a0=np.angle(np.mean(np.exp(1j*a))); b0=np.angle(np.mean(np.exp(1j*b)))
    num=np.sum(np.sin(a-a0)*np.sin(b-b0))
    den=np.sqrt(np.sum(np.sin(a-a0)**2)*np.sum(np.sin(b-b0)**2))
    return float(num/max(den,1e-12))

def wrap(a): return (a+np.pi)%(2*np.pi)-np.pi
def octant(th): return np.floor((th%(2*np.pi))/(2*np.pi/8)).astype(int)   # 0..7 (self-consistent)

def make_probe(kind):
    if kind=='linear': return Ridge(alpha=1.0)
    return MLPRegressor(hidden_layer_sizes=(64,64), max_iter=1500, alpha=1e-3, random_state=0)

def eval_phase(Zs, kind, theta_t):
    S=np.c_[np.cos(theta_t), np.sin(theta_t)]
    m=make_probe(kind).fit(Zs[tr&active], S[tr&active])
    P=m.predict(Zs); th_hat=np.arctan2(P[:,1],P[:,0])
    b=va&active
    cc=circ_corr(th_hat[b], theta_t[b])
    acc=float(balanced_accuracy_score(octant(theta_t[b]), octant(th_hat[b])))
    ang=float(np.degrees(np.mean(np.abs(wrap(th_hat[b]-theta_t[b])))))
    return th_hat, cc, acc, ang

def eval_amp(Zs, kind, A_t):
    m=make_probe(kind).fit(Zs[tr&active], A_t[tr&active])
    Ah=m.predict(Zs); b=va&active
    return Ah, float(r2_score(A_t[b], Ah[b]))

def eval_latent(Zaligned, tag, theta_t=None, A_t=None):
    if theta_t is None: theta_t=theta_true
    if A_t   is None: A_t   =A_true
    Zs=StandardScaler().fit(Zaligned[tr&active]).transform(Zaligned)
    rows=[]; store={}
    for kind in ['linear','mlp']:
        th_hat,cc,acc,ang=eval_phase(Zs,kind,theta_t)
        Ah,ar=eval_amp(Zs,kind,A_t)
        rows.append({'latent':tag,'dim':int(Zaligned.shape[1]),'probe':kind,
                     'amp_R2':round(ar,3),'phase_circ_corr':round(cc,3),
                     'phase_8sec_acc':round(acc,3),'mean_ang_err_deg':round(ang,1)})
        store[kind]=(th_hat,Ah)
    return rows, store
print('probes ready')

## Cell 3 — Run (own-RMM target): ceiling + Barlow D3/D7, table + bar chart

In [ ]:
allrows=[]; STORE={}
# pipeline ceiling: own-RMM PCs reconstructing their OWN phase/amplitude
r,_=eval_latent(own_k.copy(),'own-RMM(self ceiling)'); allrows+=r
# the Barlow latents
for name in BARLOW:
    Z=load_barlow(name)
    if Z is None: continue
    r,s=eval_latent(Z,name); allrows+=r; STORE[name]=(Z,s)

comp=pd.DataFrame(allrows); print(comp.to_string(index=False))
comp.to_csv(f'{OUT}/rmm_reconstruction.csv', index=False)
print('\nreferences (chance): phase_circ_corr~0, 8sec_acc~0.125, ang_err~90 deg, amp_R2~0')
print('note: linear self-ceiling < 1 is EXPECTED (cos/sin & sqrt are nonlinear in RMM1,RMM2); MLP ceiling ~1.')

bar=comp[comp.latent.str.startswith('Barlow')].reset_index(drop=True)
lab=[f"{r.latent}\n{r.probe}" for r in bar.itertuples()]; xs=np.arange(len(bar))
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
ax[0].bar(xs, bar['amp_R2'], color='#1f77b4'); ax[0].axhline(0,color='k',lw=.6)
ax[0].set_title('Amplitude reconstruction R2'); ax[0].set_xticks(xs); ax[0].set_xticklabels(lab,fontsize=8)
ax[1].bar(xs, bar['phase_circ_corr'], color='#d62728')
ax[1].axhline(0,color='k',lw=.6,label='chance'); ax[1].axhline(1,color='g',ls=':',label='perfect')
ax[1].set_ylim(-0.15,1.05); ax[1].set_title('Phase reconstruction (circular corr)')
ax[1].set_xticks(xs); ax[1].set_xticklabels(lab,fontsize=8); ax[1].legend(fontsize=8)
fig.suptitle('Barlow -> RMM: amplitude vs phase (own-RMM target)',fontweight='bold')
plt.tight_layout(); p=f'{OUT}/barlow_rmm_bars.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 4 — Trajectory figure: true vs Barlow-reconstructed RMM

Reconstructed index `(RMM1_hat, RMM2_hat) = A_hat * (cos theta_hat, sin theta_hat)` for the strongest
Barlow latent (D7, MLP). Expectation: the **amplitude ring survives** but the **angle is scrambled**
(color = TRUE phase should NOT organize around the reconstructed ring).

In [ ]:
pick='Barlow-D7' if 'Barlow-D7' in STORE else (list(STORE)[0] if STORE else None)
if pick is None:
    print('no Barlow latent available -- skipping trajectory figure')
else:
    Z,s=STORE[pick]; th_hat,Ah=s['mlp']; b=va&active
    R1h=Ah*np.cos(th_hat); R2h=Ah*np.sin(th_hat)
    fig,ax=plt.subplots(1,3,figsize=(16,5))
    for a in ax[:2]:
        th=np.linspace(0,2*np.pi,100); a.plot(np.cos(th),np.sin(th),'k-',lw=.5,alpha=.4); a.set_aspect('equal')
    ax[0].scatter(RMM1[b],RMM2[b],c=octant(theta_true[b]),cmap='hsv',s=6)
    ax[0].set_title('True own-RMM (color = phase)'); ax[0].set_xlabel('RMM1'); ax[0].set_ylabel('RMM2')
    ax[1].scatter(R1h[b],R2h[b],c=octant(theta_true[b]),cmap='hsv',s=6)
    ax[1].set_title(f'{pick} reconstruction (MLP)\ncolor = TRUE phase'); ax[1].set_xlabel('RMM1_hat'); ax[1].set_ylabel('RMM2_hat')
    ax[2].scatter(np.degrees(theta_true[b]),np.degrees(th_hat[b]),s=5,alpha=.35)
    ax[2].set_xlabel('true phase angle (deg)'); ax[2].set_ylabel('predicted phase angle (deg)')
    ax[2].set_title('predicted vs true angle\n(diffuse = phase not recovered)')
    plt.tight_layout(); p=f'{OUT}/barlow_rmm_trajectory.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

    fig,ax=plt.subplots(figsize=(5,5))
    ax.scatter(A_true[b],Ah[b],s=6,alpha=.35); lim=[0,float(max(A_true[b].max(),Ah[b].max()))]
    ax.plot(lim,lim,'k--',lw=.6); ax.set_xlabel('true amplitude'); ax.set_ylabel('predicted amplitude')
    ax.set_title(f'{pick} amplitude recovery (MLP)')
    plt.tight_layout(); p=f'{OUT}/barlow_rmm_amp_scatter.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 5 — Robustness check: repeat with OFFICIAL BoM RMM as target

In [ ]:
A_bom=np.sqrt(bom_k[:,0]**2+bom_k[:,1]**2); th_bom=np.arctan2(bom_k[:,1],bom_k[:,0])
bomrows=[]
for name in BARLOW:
    Z=load_barlow(name)
    if Z is None: continue
    r,_=eval_latent(Z,name,theta_t=th_bom,A_t=A_bom); bomrows+=r
bom_df=pd.DataFrame(bomrows); print('OFFICIAL BoM RMM target:'); print(bom_df.to_string(index=False))
bom_df.to_csv(f'{OUT}/rmm_reconstruction_bom.csv',index=False)

merge=comp[comp.latent.str.startswith('Barlow')].merge(bom_df,on=['latent','probe'],suffixes=('_own','_bom'))
print('\nown vs BoM (Barlow) — should agree if the split is robust:')
print(merge[['latent','probe','amp_R2_own','amp_R2_bom','phase_circ_corr_own','phase_circ_corr_bom']].to_string(index=False))

## Cell 6 — Summary JSON + read-out / interpretation

In [ ]:
summary={'purpose':'quantify phase/envelope split via RMM reconstruction from Barlow latents',
 'target':'own-RMM (primary) + official BoM (check)','probes':['linear Ridge','MLP'],
 'n_active':int(active.sum()),'n_val_active':int((va&active).sum()),
 'own_target':comp.to_dict(orient='records'),
 'bom_target':bom_df.to_dict(orient='records'),
 'references':{'phase_circ_corr_chance':0.0,'phase_8sec_acc_chance':0.125,'ang_err_chance_deg':90.0,'amp_R2_chance':0.0}}
json.dump(summary, open(f'{OUT}/barlow_rmm_summary.json','w'), indent=2, default=float)
print('Saved', f'{OUT}/barlow_rmm_summary.json')
print('\n=== READ-OUT (own-RMM target, val&active) ===')
for r in comp[comp.latent.str.startswith('Barlow')].itertuples():
    print(f'{r.latent:10s} {r.probe:6s} amp_R2={r.amp_R2:+.3f}  phase_circ_corr={r.phase_circ_corr:+.3f}'
          f'  8sec_acc={r.phase_8sec_acc:.3f}  ang_err={r.mean_ang_err_deg:.0f} deg')
print('\nInterpretation:')
print('  amp_R2 > 0 AND phase_circ_corr ~ 0 (8sec~0.125, ang_err~90) ->')
print('    Barlow encodes the RMM AMPLITUDE envelope, NOT the phase clock (confirms summary sec 8).')
print('  MLP phase_circ_corr > ~0.4 -> phase was nonlinearly hidden -> revisit sec 8.')

---
## Done!
**Send back:** the Cell 3 table + `barlow_rmm_bars.png`, the Cell 4 trajectory PNG, and the Cell 5 own-vs-BoM comparison.

*DDCS Project | jh9141@nyu.edu*